# Ablation Study

This notebook presents a systematic investigation of the factors driving
performance differences between the three evaluated models.

Investigation 1: AnomalyDINO 2x2 Factorial Ablation
A controlled factorial design isolating the two hypothesised failure
modes of AnomalyDINO on Real-IAD: intra-class viewpoint variation
and cross-category feature contamination. Four conditions are evaluated
on five representative categories.

Investigation 2: Training Compute Equalisation (Dinomaly vs INP-Former)
Dinomaly and INP-Former use fundamentally different training schedules
resulting in a roughly 9x disparity in total image passes. This
investigation equalises compute budgets to assess whether performance
differences reflect architecture or training volume.

Investigation 3: Cross-View Training Data Volume Compensation
The cross-view protocol reduces training data to approximately 2/5
of the standard protocol. This investigation tests whether scaling
the training budget proportionally recovers the performance gap for
Dinomaly and INP-Former.

Five representative categories are used for Investigation 1:
audiojack, pcb, button_battery, usb, toothbrush
Selected to represent diversity in object geometry, defect type,
and detection difficulty across Real-IAD.

In [ ]:
import os
os.environ['PYTORCH_ALLOC_CONF'] = 'expandable_segments:True'

from google.colab import drive
import sys

drive.mount('/content/drive')

repo_path = '/content/drive/MyDrive/BachelorsThesis'
dataset_root = '/content/drive/MyDrive/datasets/realiad_512'

if not os.path.exists(repo_path):
    !git clone https://github.com/PurpleMono/BachelorsThesis.git {repo_path}
    !git -C {repo_path} submodule update --init
else:
    !git -C {repo_path} pull
    !git -C {repo_path} submodule update --init

!git -C {repo_path}/models/inp_former fetch origin
!git -C {repo_path}/models/inp_former checkout 6041e2b

sys.path.insert(0, repo_path)

!pip install anomalib==2.4.0 ADEval einops timm kornia -q

import torch
import numpy as np
import pandas as pd
import gc
from pathlib import Path

print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
import zipfile, shutil, os

zip_dir = '/content/drive/MyDrive/datasets/realiad_512/realiad_512'
target_dir = '/content/realiad_512'
os.makedirs(target_dir, exist_ok=True)

# Copy JSON metadata (small — shutil is fine here)
json_src = '/content/drive/MyDrive/datasets/realiad_512/realiad_jsons'
json_dst = '/content/realiad_512/realiad_jsons'
if not os.path.exists(json_dst):
    shutil.copytree(json_src, json_dst)
    print("JSONs copied")
else:
    print("JSONs already present")

# Unzip each category from Drive zips
for f in sorted(os.listdir(zip_dir)):
    if f.endswith('.zip'):
        category = f.replace('.zip', '')
        if not os.path.exists(f'{target_dir}/{category}'):
            print(f"Unzipping {f}...")
            with zipfile.ZipFile(f'{zip_dir}/{f}', 'r') as z:
                z.extractall(target_dir)
        else:
            print(f"Skipping {category} — already present")

dataset_root = target_dir
print(f"\nDataset ready at: {dataset_root}")

In [ ]:
import importlib.util

def load_module(name, path):
    spec = importlib.util.spec_from_file_location(name, path)
    mod = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(mod)
    return mod

realiad_utils = load_module("realiad_utils", f"{repo_path}/data/realiad_utils.py")
trainer = load_module("trainer", f"{repo_path}/models/trainer.py")
metrics = load_module("metrics", f"{repo_path}/evaluation/metrics.py")

load_realiad_category = realiad_utils.load_realiad_category
load_realiad_all = realiad_utils.load_realiad_all
get_crossview_split = realiad_utils.get_crossview_split
train_anomalydino = trainer.train_anomalydino
train_dinomaly = trainer.train_dinomaly
train_inpformer = trainer.train_inpformer
run_inference = trainer.run_inference
run_inference_inpformer = trainer.run_inference_inpformer
compute_i_auroc = metrics.compute_i_auroc
compute_s_auroc = metrics.compute_s_auroc
compute_degradation_ratio = metrics.compute_degradation_ratio

print("All modules loaded")

In [ ]:
results_path = f'{repo_path}/results'

# Load standard protocol baselines
results_din_std = pd.read_csv(
    f'{results_path}/dinomaly_standard_scores.csv')
results_dino_std = pd.read_csv(
    f'{results_path}/anomalydino_standard_scores.csv')
results_inp_std = pd.read_csv(
    f'{results_path}/inpformer_standard_scores.csv')

# Load cross-view protocol baselines
results_din_cv = pd.read_csv(
    f'{results_path}/dinomaly_crossview_scores.csv')
results_inp_cv = pd.read_csv(
    f'{results_path}/inpformer_crossview_scores.csv')

i_auroc_din_std = compute_i_auroc(results_din_std)
i_auroc_dino_std = compute_i_auroc(results_dino_std)
i_auroc_inp_std = compute_i_auroc(results_inp_std)
i_auroc_din_cv = compute_i_auroc(results_din_cv)
i_auroc_inp_cv = compute_i_auroc(results_inp_cv)

print("Standard protocol baselines:")
print(f"  Dinomaly:    {i_auroc_din_std:.4f}")
print(f"  AnomalyDINO: {i_auroc_dino_std:.4f}")
print(f"  INP-Former:  {i_auroc_inp_std:.4f}")
print("\nCross-view baselines:")
print(f"  Dinomaly:    {i_auroc_din_cv:.4f}")
print(f"  INP-Former:  {i_auroc_inp_cv:.4f}")

## Investigation 1: AnomalyDINO 2x2 Factorial Ablation

AnomalyDINO fails on Real-IAD with near-random I-AUROC (~0.50) under
the standard multi-view multi-class protocol. Two hypotheses are
investigated as potential causes:

Hypothesis A: Cross-category feature contamination
In the multi-class setting a global memory bank stores features from
all 30 categories. Anomalous patches may find false nearest neighbours
in normal features from unrelated categories, suppressing anomaly scores.

Hypothesis B: Intra-class viewpoint variation
Real-IAD captures each object from five camera angles. Normal patches
from one viewpoint may be dissimilar to normal patches from other
viewpoints in the same category, producing high distances even for
normal regions and collapsing the anomaly signal.

Four conditions on five representative categories isolate each factor:

Condition A — Multi-Class, Multi-View (standard protocol result, already done)
All 30 categories in memory bank, all 5 viewpoints for train and test.

Condition B — Single-Class, Multi-View
One category in memory bank, all 5 viewpoints for train and test.
Tests whether removing cross-category contamination helps.

Condition C — Multi-Class, Single-View
All 30 categories in memory bank, C1 only for train and test.
Tests whether removing viewpoint variation helps when multi-class.

Condition D — Single-Class, Single-View
One category in memory bank, C1 only for train and test.
Best case for AnomalyDINO — isolates both confounds simultaneously.

If B improves significantly over A: cross-category contamination matters.
If C improves significantly over A: viewpoint variation matters.
If D is the only strong performer: both factors are necessary.

In [ ]:
# Five representative categories
# Selected for diversity in geometry, defect type, and difficulty
ABL_CATEGORIES = [
    'audiojack',      # multi-part connector, multiple defect types
    'pcb',            # complex circuit board, high defect diversity
    'button_battery', # simple round object, clean geometry
    'usb',            # elongated connector, different geometry
    'toothbrush',     # simple single object, low complexity
]

# Load all category data upfront
print("Loading ablation categories...")
cat_data = {}
for cat in ABL_CATEGORIES:
    df = load_realiad_category(
        category_root=f'{dataset_root}/{cat}',
        json_path=f'{dataset_root}/realiad_jsons/{cat}.json'
    )
    cat_data[cat] = df
    print(f"  {cat}: {len(df)} images")

# Load all 30 categories for multi-class conditions
print("\nLoading all categories for multi-class conditions...")
df_all = load_realiad_all(data_root=dataset_root)
print(f"  Total: {len(df_all)} images across {df_all['category'].nunique()} categories")

In [ ]:
print("="*60)
print("CONDITION B: Single-Class, Multi-View")
print("One category memory bank, all 5 viewpoints")
print("="*60)

cond_b_results = {}

for cat in ABL_CATEGORIES:
    print(f"\nCategory: {cat}")
    df = cat_data[cat]

    # Train on all viewpoints, one category only
    train_df = df[(df['split']=='train') & (df['label']==0)]
    test_df = df[df['split']=='test']

    model = train_anomalydino(
        train_df=train_df,
        device='cuda',
        repo_path=repo_path,
        sampling_ratio=0.1
    )

    results = run_inference(
        model=model,
        test_df=test_df,
        model_name='AnomalyDINO',
        device='cuda',
        batch_size=8,
        repo_path=repo_path
    )

    i_auroc = compute_i_auroc(results)
    cond_b_results[cat] = i_auroc
    print(f"  I-AUROC: {i_auroc:.4f}")

    torch.cuda.empty_cache()
    gc.collect()
    del model

print("\nCondition B complete")
print(cond_b_results)

In [ ]:
print("="*60)
print("CONDITION C: Multi-Class, Single-View")
print("All 30 categories memory bank, C1 viewpoint only")
print("="*60)

# Build one global multi-class memory bank from C1 only
train_df_c1_all = df_all[
    (df_all['split']=='train') &
    (df_all['label']==0) &
    (df_all['viewpoint']=='C1')
].reset_index(drop=True)

print(f"Training on C1 only across all categories: "
      f"{len(train_df_c1_all)} normal images")

model_mc_sv = train_anomalydino(
    train_df=train_df_c1_all,
    device='cuda',
    repo_path=repo_path,
    sampling_ratio=0.1
)

cond_c_results = {}

for cat in ABL_CATEGORIES:
    print(f"\nEvaluating: {cat}")
    df = cat_data[cat]

    # Test on C1 viewpoint only
    test_df_c1 = df[
        (df['split']=='test') &
        (df['viewpoint']=='C1')
    ].reset_index(drop=True)

    results = run_inference(
        model=model_mc_sv,
        test_df=test_df_c1,
        model_name='AnomalyDINO',
        device='cuda',
        batch_size=8,
        repo_path=repo_path
    )

    i_auroc = compute_i_auroc(results)
    cond_c_results[cat] = i_auroc
    print(f"  I-AUROC (C1 only): {i_auroc:.4f}")

torch.cuda.empty_cache()
gc.collect()
del model_mc_sv
print("\nCondition C complete")

In [ ]:
print("="*60)
print("CONDITION D: Single-Class, Single-View")
print("One category memory bank, C1 viewpoint only")
print("Best case for AnomalyDINO")
print("="*60)

cond_d_results = {}

for cat in ABL_CATEGORIES:
    print(f"\nCategory: {cat}")
    df = cat_data[cat]

    # Train on C1 only, one category
    train_df_c1 = df[
        (df['split']=='train') &
        (df['label']==0) &
        (df['viewpoint']=='C1')
    ].reset_index(drop=True)

    # Test on C1 only
    test_df_c1 = df[
        (df['split']=='test') &
        (df['viewpoint']=='C1')
    ].reset_index(drop=True)

    print(f"  Train: {len(train_df_c1)} | Test: {len(test_df_c1)}")

    model = train_anomalydino(
        train_df=train_df_c1,
        device='cuda',
        repo_path=repo_path,
        sampling_ratio=0.1
    )

    results = run_inference(
        model=model,
        test_df=test_df_c1,
        model_name='AnomalyDINO',
        device='cuda',
        batch_size=8,
        repo_path=repo_path
    )

    i_auroc = compute_i_auroc(results)
    cond_d_results[cat] = i_auroc
    print(f"  I-AUROC: {i_auroc:.4f}")

    torch.cuda.empty_cache()
    gc.collect()
    del model

print("\nCondition D complete")

In [ ]:
# Condition A: Multi-Class, Multi-View — from standard protocol results
cond_a_results = {}
for cat in ABL_CATEGORIES:
    cat_df = results_dino_std[results_dino_std['category']==cat]
    if len(cat_df) > 0:
        cond_a_results[cat] = compute_i_auroc(cat_df)
    else:
        cond_a_results[cat] = float('nan')

# Build summary table
abl1_rows = []
for cat in ABL_CATEGORIES:
    a = cond_a_results.get(cat, float('nan'))
    b = cond_b_results.get(cat, float('nan'))
    c = cond_c_results.get(cat, float('nan'))
    d = cond_d_results.get(cat, float('nan'))
    abl1_rows.append({
        'Category': cat,
        'A: MC+MV (standard)': round(a, 4),
        'B: SC+MV': round(b, 4),
        'C: MC+SV': round(c, 4),
        'D: SC+SV (best case)': round(d, 4),
        'B-A (contamination effect)': round(b-a, 4),
        'C-A (viewpoint effect)': round(c-a, 4),
        'D-A (combined effect)': round(d-a, 4),
    })

abl1_df = pd.DataFrame(abl1_rows)

# Add mean row
means = abl1_df.mean(numeric_only=True)
means['Category'] = 'Mean'
abl1_df = pd.concat(
    [abl1_df, pd.DataFrame([means])], ignore_index=True)

print("="*70)
print("INVESTIGATION 1: AnomalyDINO 2x2 FACTORIAL ABLATION")
print("MC=Multi-Class, SC=Single-Class, MV=Multi-View, SV=Single-View")
print("="*70)
print(abl1_df.round(4).to_string(index=False))

os.makedirs(f'{results_path}', exist_ok=True)
abl1_df.to_csv(
    f'{results_path}/ablation1_factorial_summary.csv', index=False)
print(f"\nSaved to results/ablation1_factorial_summary.csv")

print("\nInterpretation:")
mean_b_effect = abl1_df[abl1_df['Category']!='Mean']['B-A (contamination effect)'].mean()
mean_c_effect = abl1_df[abl1_df['Category']!='Mean']['C-A (viewpoint effect)'].mean()
print(f"  Mean contamination effect (B-A): {mean_b_effect:+.4f}")
print(f"  Mean viewpoint effect (C-A):     {mean_c_effect:+.4f}")
if abs(mean_c_effect) > abs(mean_b_effect):
    print("  Viewpoint variation is the dominant failure mode")
else:
    print("  Cross-category contamination is the dominant failure mode")

## Investigation 2: Training Compute Equalisation

Dinomaly2 uses 100,000 iterations with batch size 16 resulting in
approximately 1,600,000 total image passes on Real-IAD.

INP-Former uses 200 epochs resulting in approximately 7,293,000
total image passes — roughly 4,5x more than Dinomaly.

This investigation trains  INP-Former model at an equalised compute budget
of approximately 1,600,000 image passes to assess whether observed
performance differences reflect architecture or training volume.

Equalised budgets:
- INP-Former: 44 epochs (~1,6M image passes)

Both models are evaluated on the standard protocol (all 30 categories,
all 5 viewpoints) to isolate the effect of compute from viewpoint shift.

In [ ]:
print("="*60)
print("INVESTIGATION 2: COMPUTE EQUALISATION (INP-Former only)")
print("="*60)

# Dinomaly: 100,000 iterations x batch 16 = 1,600,000 image passes
# INP-Former equalised: 1,600,000 / 36,465 images = ~44 epochs
# Dinomaly excluded from retraining — standard protocol result
# (100k iterations) serves as its baseline directly
EQUALISED_EPOCHS_INPFORMER = 44

# Load full training and test sets
train_df_std = df_all[
    (df_all['split'] == 'train') & (df_all['label'] == 0)]
test_df_std = df_all[df_all['split'] == 'test']

print(f"INP-Former equalised epochs: {EQUALISED_EPOCHS_INPFORMER}")
print(f"(~1,600,000 image passes — matches Dinomaly standard protocol)")
print(f"Training images: {len(train_df_std)}")

model_inp_eq = train_inpformer(
    train_df=train_df_std,
    dataset_root=dataset_root,
    n_epochs=EQUALISED_EPOCHS_INPFORMER,
    batch_size=16,
    device='cuda',
    repo_path=repo_path,
    save_path=f'{results_path}/weights/inpformer_equalised.pth'
)

results_inp_eq = run_inference_inpformer(
    model=model_inp_eq,
    test_df=test_df_std,
    dataset_root=dataset_root,
    device='cuda',
    batch_size=16,
    repo_path=repo_path,
    save_anomaly_maps=False
)

os.makedirs(results_path, exist_ok=True)
results_inp_eq.to_csv(
    f'{results_path}/inpformer_equalised_scores.csv', index=False)
print(f"Saved: {len(results_inp_eq)} rows")

i_auroc_inp_eq = compute_i_auroc(results_inp_eq)
i_auroc_inp_std = compute_i_auroc(
    pd.read_csv(f'{results_path}/inpformer_standard_scores.csv'))
i_auroc_din_std = compute_i_auroc(
    pd.read_csv(f'{results_path}/dinomaly_standard_scores.csv'))

print(f"\nCOMPUTE EQUALISATION RESULTS")
print(f"Dinomaly standard (100k iter, ~1.6M passes):    {i_auroc_din_std:.4f}")
print(f"INP-Former standard (200 epochs, ~7.3M passes): {i_auroc_inp_std:.4f}")
print(f"INP-Former equalised (44 epochs, ~1.6M passes): {i_auroc_inp_eq:.4f}")
print(f"Delta (equalised vs standard): "
      f"{i_auroc_inp_eq - i_auroc_inp_std:+.4f}")

torch.cuda.empty_cache()
gc.collect()
del model_inp_eq

In [ ]:
abl2_summary = pd.DataFrame({
    'Model': ['Dinomaly', 'INP-Former'],
    'Standard I-AUROC': [i_auroc_din_std, i_auroc_inp_std],
    'Standard Image Passes': [800000, 7293000],
    'Equalised I-AUROC': [i_auroc_din_eq, i_auroc_inp_eq],
    'Equalised Image Passes': [3000000, 3000000],
    'Delta': [
        i_auroc_din_eq - i_auroc_din_std,
        i_auroc_inp_eq - i_auroc_inp_std,
    ],
})

print("="*70)
print("INVESTIGATION 2: COMPUTE EQUALISATION RESULTS")
print("="*70)
print(abl2_summary.round(4).to_string(index=False))

abl2_summary.to_csv(
    f'{results_path}/ablation2_compute_summary.csv', index=False)
print(f"\nSaved to results/ablation2_compute_summary.csv")

## Investigation 3: Cross-View Training Data Volume Compensation (INP-Former)

The cross-viewpoint protocol trains on C1 and C2 only, reducing the
training set to approximately 2/5 of the standard protocol size.

For Dinomaly, this has no effect on total compute — iteration-based
training processes a fixed number of image passes (50,000 iterations
x batch size 16 = 800,000 passes) regardless of dataset size. Any
performance degradation for Dinomaly under the cross-view protocol
is therefore attributable purely to reduced viewpoint diversity, not
reduced training volume.

For INP-Former, epoch-based training means fewer total image passes
when the dataset is smaller:
- Standard protocol: 200 epochs x ~36,465 images = ~7,293,000 passes
- Cross-view protocol: 200 epochs x ~14,586 images = ~2,917,200 passes

This represents a ~2.5x reduction in total image passes for INP-Former
under the cross-view protocol. This investigation compensates by
scaling epochs proportionally:
- Compensated: 500 epochs x ~14,586 images = ~7,293,000 passes

If performance recovers with 500 epochs, the cross-view degradation
for INP-Former is attributable to reduced training volume. If it does
not recover, viewpoint coverage itself is the primary factor.

In [ ]:
# Load cross-view baseline results
results_din_cv = pd.read_csv(
    f'{repo_path}/results/dinomaly_crossview_scores.csv')
results_inp_cv = pd.read_csv(
    f'{repo_path}/results/inpformer_crossview_scores.csv')

i_auroc_din_cv = compute_i_auroc(results_din_cv)
i_auroc_inp_cv = compute_i_auroc(results_inp_cv)

# Load standard protocol results
results_din_std = pd.read_csv(
    f'{repo_path}/results/dinomaly_standard_scores.csv')
results_inp_std = pd.read_csv(
    f'{repo_path}/results/inpformer_standard_scores.csv')

i_auroc_din_std = compute_i_auroc(results_din_std)
i_auroc_inp_std = compute_i_auroc(results_inp_std)

print("Cross-view baselines:")
print(f"  Dinomaly:    {i_auroc_din_cv:.4f}")
print(f"  INP-Former:  {i_auroc_inp_cv:.4f}")

print("\nStandard protocol baselines:")
print(f"  Dinomaly:    {i_auroc_din_std:.4f}")
print(f"  INP-Former:  {i_auroc_inp_std:.4f}")

print("\nNote: Dinomaly excluded from volume compensation.")
print("Its iteration-based schedule already matches standard")
print("protocol total image passes under cross-view conditions.")

# Load cross-view training data
train_df_cv, test_df_cv = get_crossview_split(
    df_all,
    train_views=['C1', 'C2'],
    test_views=['C3', 'C4', 'C5']
)
train_df_cv = train_df_cv[
    train_df_cv['label'] == 0].reset_index(drop=True)

n_std = len(df_all[
    (df_all['split'] == 'train') & (df_all['label'] == 0)])
n_cv = len(train_df_cv)

print(f"\nStandard training images: {n_std}")
print(f"Cross-view training images: {n_cv}")
print(f"Ratio: {n_cv/n_std:.2f}")

std_passes_inp = 200 * n_std
cv_passes_inp = 200 * n_cv
compensated_epochs = round(std_passes_inp / n_cv)

print(f"\nINP-Former image passes:")
print(f"  Standard (200 epochs): {std_passes_inp:,}")
print(f"  Cross-view (200 epochs): {cv_passes_inp:,}")
print(f"  Compensated epochs needed: {compensated_epochs}")

In [ ]:
# Compute compensated epochs dynamically based on actual dataset sizes
compensated_epochs = round(
    (200 * len(df_all[
        (df_all['split'] == 'train') & (df_all['label'] == 0)
    ])) / len(train_df_cv)
)

print(f"Training INP-Former with {compensated_epochs} epochs")
print(f"(compensates for cross-view dataset reduction)")

model_inp_comp = train_inpformer(
    train_df=train_df_cv,
    dataset_root=dataset_root,
    n_epochs=compensated_epochs,
    batch_size=16,
    device='cuda',
    repo_path=repo_path,
    save_path=f'{repo_path}/results/weights/inpformer_cv_compensated.pth'
)

results_inp_comp = run_inference_inpformer(
    model=model_inp_comp,
    test_df=test_df_cv,
    dataset_root=dataset_root,
    device='cuda',
    batch_size=8,
    repo_path=repo_path
)

results_inp_comp.to_csv(
    f'{repo_path}/results/inpformer_cv_compensated_scores.csv', index=False)
i_auroc_inp_comp = compute_i_auroc(results_inp_comp)

print(f"\nINP-Former standard (200 epochs):             {i_auroc_inp_std:.4f}")
print(f"INP-Former cross-view (200 epochs):           {i_auroc_inp_cv:.4f}")
print(f"INP-Former cross-view ({compensated_epochs} epochs): {i_auroc_inp_comp:.4f}")
print(f"Recovery: {i_auroc_inp_comp - i_auroc_inp_cv:+.4f}")

torch.cuda.empty_cache()
gc.collect()
del model_inp_comp

In [ ]:
# Dinomaly cross-view note — no compensation needed
deg_din = compute_degradation_ratio(i_auroc_din_std, i_auroc_din_cv)
deg_inp = compute_degradation_ratio(i_auroc_inp_std, i_auroc_inp_cv)
deg_inp_comp = compute_degradation_ratio(i_auroc_inp_std, i_auroc_inp_comp)

abl3_summary = pd.DataFrame([
    {
        'Model': 'Dinomaly',
        'Standard': round(i_auroc_din_std, 4),
        'Cross-View (200 epochs equiv)': round(i_auroc_din_cv, 4),
        'Cross-View Compensated': 'N/A (iteration-based)',
        'Degradation (%)': round(deg_din, 2),
        'Recovery after compensation': 'N/A',
        'Note': 'Fixed image passes regardless of dataset size'
    },
    {
        'Model': 'INP-Former',
        'Standard': round(i_auroc_inp_std, 4),
        'Cross-View (200 epochs equiv)': round(i_auroc_inp_cv, 4),
        'Cross-View Compensated': round(i_auroc_inp_comp, 4),
        'Degradation (%)': round(deg_inp, 2),
        'Recovery after compensation': round(
            i_auroc_inp_comp - i_auroc_inp_cv, 4),
        'Note': f'Compensated to {compensated_epochs} epochs'
    },
])

print("=" * 70)
print("INVESTIGATION 3: CROSS-VIEW VOLUME COMPENSATION")
print("=" * 70)
print(abl3_summary.to_string(index=False))

abl3_summary.to_csv(
    f'{repo_path}/results/ablation3_volume_summary.csv', index=False)
print("\nSaved to results/ablation3_volume_summary.csv")

## Combined Ablation Summary

Synthesises findings across all three investigations.

In [ ]:
print("=" * 70)
print("ABLATION STUDY COMBINED SUMMARY")
print("=" * 70)

print("\nInvestigation 1: AnomalyDINO 2x2 Factorial")
print(abl1_df.round(4).to_string(index=False))

print("\nInvestigation 2: Compute Equalisation (Dinomaly vs INP-Former)")
print(abl2_summary.round(4).to_string(index=False))

print("\nInvestigation 3: Cross-View Volume Compensation (INP-Former only)")
print(abl3_summary.to_string(index=False))